In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm

import os, json, random, copy, time, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from concurrent.futures import ThreadPoolExecutor
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ---- paths ----
TEA_LEAF_ROOT = "/content/drive/MyDrive/Tea Leaf Dataset"
OUT           = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
RESULTS_DIR   = f"{OUT}/results"
FIGURES_DIR   = f"{OUT}/figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# ---- dataset -> local disk (PARALLEL, resumable) ----
LOCAL_DATA = "/tmp/tea_leaf_local"
N_WORKERS  = 32          # drop to 16 if Drive throttles

def _count_files(p):
    return sum(len(f) for _, _, f in os.walk(p)) if os.path.isdir(p) else 0

n_src = _count_files(TEA_LEAF_ROOT)
n_dst = _count_files(LOCAL_DATA)

if n_dst < n_src:
    t0 = time.time()
    tasks = []
    for root, _, files_ in os.walk(TEA_LEAF_ROOT):
        rel = os.path.relpath(root, TEA_LEAF_ROOT)
        dst_dir = LOCAL_DATA if rel == '.' else os.path.join(LOCAL_DATA, rel)
        os.makedirs(dst_dir, exist_ok=True)
        for f in files_:
            src = os.path.join(root, f)
            dst = os.path.join(dst_dir, f)
            if not os.path.exists(dst):          # resume: skip what's already there
                tasks.append((src, dst))

    print(f"Copying {len(tasks)} files ({n_dst}/{n_src} already local) "
          f"with {N_WORKERS} workers...")

    def _cp(t):
        try:
            shutil.copyfile(t[0], t[1]); return True
        except Exception:
            return False

    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        ok = list(tqdm(ex.map(_cp, tasks), total=len(tasks), desc="copy"))

    print(f" Copied {sum(ok)}/{len(tasks)} in {time.time()-t0:.0f}s")
else:
    print(f" Local cache complete ({n_dst} files)")

# ---- classes ----
classes = sorted([d for d in os.listdir(LOCAL_DATA)
                  if os.path.isdir(os.path.join(LOCAL_DATA, d))])
NUM_CLASSES     = len(classes)
CLASS_TO_IDX    = {c: i for i, c in enumerate(classes)}
healthy_classes = [c for c in classes if 'healthy' in c.lower()]
print(f"Classes ({NUM_CLASSES}): {classes}")

# ---- DINOv1 ViT-S/16 ----
dv1_path = os.path.join(OUT, 'checkpoints', 'dino_vits16.pth')
if not os.path.exists(dv1_path):
    os.makedirs(os.path.dirname(dv1_path), exist_ok=True)
    !wget -q -O {dv1_path} https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth

dino_v1 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
ckpt = torch.load(dv1_path, map_location='cpu')
sd = {k.replace('module.','').replace('backbone.','').replace('encoder.',''): v
      for k, v in ckpt.items() if 'head' not in k}
dino_v1.load_state_dict(sd, strict=False)
dino_v1.eval().to(device)
for p in dino_v1.parameters():
    p.requires_grad = False

# ---- attention hook ----
_dv1_store = {}
dv1_last_attn = dino_v1.blocks[-1].attn

def _dv1_patched_forward(x, **kwargs):
    B, N, C = x.shape
    qkv = dv1_last_attn.qkv(x).reshape(B, N, 3, dv1_last_attn.num_heads,
                                       C // dv1_last_attn.num_heads)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    scale = (C // dv1_last_attn.num_heads) ** -0.5
    cls_q = q[:, :, 0:1, :]
    cls_attn = ((cls_q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    _dv1_store['cls_attn'] = cls_attn[:, :, 0, 1:].detach().cpu()
    attn_full = ((q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    x = (attn_full @ v).transpose(1, 2).reshape(B, N, C)
    x = dv1_last_attn.proj(x)
    x = dv1_last_attn.proj_drop(x)
    return x

dv1_last_attn.forward = _dv1_patched_forward

# ---- transforms ----
tf_dv1 = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
tf_train = T.Compose([
    T.Resize((224, 224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(20), T.ColorJitter(0.2, 0.2, 0.2),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
tf_test = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

DV1_GRID = 14
N_HEADS  = 6
topk     = max(1, int(0.1 * DV1_GRID * DV1_GRID))

with torch.no_grad():
    _ = dino_v1(torch.randn(1, 3, 224, 224).to(device))
print(f"Attention hook OK: {_dv1_store['cls_attn'].shape}")

# ---- image list ----
all_images = []
for cls_name in classes:
    cls_dir = os.path.join(LOCAL_DATA, cls_name)
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_images.append({
                'path': os.path.join(cls_dir, fname),
                'class': cls_name,
                'class_idx': CLASS_TO_IDX[cls_name],
                'is_diseased': 0 if cls_name in healthy_classes else 1,
            })

# ---- datasets ----
class SimpleDS(Dataset):
    def __init__(self, imgs, labels, transform):
        self.imgs = imgs; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]['path']).convert('RGB')
        return self.transform(img), self.labels[idx]

class TestDS(Dataset):
    def __init__(self, imgs, transform):
        self.imgs = imgs; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]['path']).convert('RGB')
        return self.transform(img), self.imgs[idx]['is_diseased']

n_dis = sum(x['is_diseased'] for x in all_images)
print(f"\n Ready: {len(all_images)} images | Diseased: {n_dis}, Healthy: {len(all_images)-n_dis}")

In [ ]:
!rm -f /content/leakage_audit.py

In [ ]:
import os, shutil, sys
from google.colab import files

if not os.path.exists("/content/leakage_audit.py"):
    print("Select leakage_audit.py (the UPDATED version):")
    up = files.upload()
    src = list(up.keys())[0]
    if os.path.abspath(src) != "/content/leakage_audit.py":
        shutil.move(src, "/content/leakage_audit.py")

# Read the content of the file
with open("/content/leakage_audit.py", "r") as f:
    lines = f.readlines()

# Filter out lines that start with '!' (shell commands)
cleaned_lines = [line for line in lines if not line.strip().startswith('!')]

# Overwrite the file with the cleaned content
with open("/content/leakage_audit.py", "w") as f:
    f.writelines(cleaned_lines)

sys.modules.pop("leakage_audit", None)
import leakage_audit as la

assert hasattr(la, "inspect_cross_class"), \
    "Old version on disk. Run  !rm /content/leakage_audit.py  and re-run this cell."
print("Module OK:", la.__file__)

In [ ]:
report, uf, imgs = la.run_audit(
    data_root      = LOCAL_DATA,
    seed           = 42, test_frac = 0.2,
    phash_thresh   = 0, dhash_thresh = 0,        # exact-canonical only
    use_embeddings = True, emb_cos_thresh = 0.97,
    out_json       = os.path.join(RESULTS_DIR, "leakage_audit.json"),
)
assert getattr(uf, "provenance", None) is not None, "old run_audit ran — fix G1"
print("\nuf + imgs ready")

In [ ]:
import os, shutil, sys
from google.colab import files

if not os.path.exists("/content/triage_analysis.py"):
    print("Select triage_analysis.py:")
    up = files.upload()
    src = list(up.keys())[0]
    if os.path.abspath(src) != "/content/triage_analysis.py":
        shutil.move(src, "/content/triage_analysis.py")

sys.modules.pop("triage_analysis", None)
import triage_analysis as ta
print("OK:", ta.__file__)

In [ ]:
import os, shutil
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# 1. verify every local image actually opens
def _bad(rec):
    try:
        Image.open(rec['path']).convert('RGB')
        return None
    except Exception:
        return rec['path']

with ThreadPoolExecutor(max_workers=16) as ex:
    results = list(tqdm(ex.map(_bad, all_images), total=len(all_images), desc="verify"))
bad = [p for p in results if p]
print(f"\nCorrupt/unreadable: {len(bad)} of {len(all_images)}")

# 2. re-copy the bad ones from Drive, one at a time (no parallelism)
fixed, still_bad = 0, []
for p in bad:
    rel = os.path.relpath(p, LOCAL_DATA)
    src = os.path.join(TEA_LEAF_ROOT, rel)
    try:
        os.remove(p)
        shutil.copyfile(src, p)
        Image.open(p).convert('RGB')     # verify the re-copy
        fixed += 1
    except Exception as e:
        still_bad.append(p)
        print(f"  FAILED: {rel}  ({e})")

print(f"\nRe-copied OK : {fixed}")
print(f"Still bad    : {len(still_bad)}")

# 3. if any file is bad on Drive itself, drop it from the dataset
if still_bad:
    before = len(all_images)
    bad_set = set(still_bad)
    all_images = [x for x in all_images if x['path'] not in bad_set]
    print(f"\nDropped {before - len(all_images)} genuinely-corrupt images "
          f"from Drive; {len(all_images)} remain.")
    print("NOTE: `uf`/`imgs` from the audit still reference the old list —")
    print("re-run G2 before D1 so the split matches.")
else:
    print("\nAll images valid. Re-run D1.")

In [ ]:
import numpy as np, random, json, os, torch, torch.nn as nn, torch.optim as optim
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from PIL import Image
from tqdm import tqdm
import leakage_audit as la

assert 'uf' in dir() and 'imgs' in dir(), "Run the audit cell (G2) first"

SEED = 0
random.seed(SEED*1000 + 42)
train_s, test_s = la.grouped_split(uf, imgs, seed=SEED*1000+42, test_frac=0.2)

# --- UAHS head selection ---
pool = random.sample(train_s, min(200, len(train_s)))
head_ent = {h: [] for h in range(N_HEADS)}
with torch.no_grad():
    for s in pool:
        try:
            im = Image.open(s['path']).convert('RGB')
            _ = dino_v1(tf_dv1(im).unsqueeze(0).to(device))
            attn = _dv1_store['cls_attn'][0].numpy()
            for h in range(N_HEADS):
                a = attn[h]; a = a/(a.sum()+1e-10)
                head_ent[h].append(-(a*np.log(a+1e-10)).sum())
        except Exception:
            continue
sel_heads = sorted(range(N_HEADS), key=lambda h: -np.std(head_ent[h]))[:2]
print(f"UAHS heads: {sel_heads}")

# --- score + pseudo-label ---
scores = []
with torch.no_grad():
    for s in tqdm(train_s, desc="Scoring"):
        try:
            im = Image.open(s['path']).convert('RGB')
            _ = dino_v1(tf_dv1(im).unsqueeze(0).to(device))
            attn = _dv1_store['cls_attn'][0].numpy()
            a = attn[sel_heads].mean(axis=0); a = a/(a.sum()+1e-10)
            scores.append(np.sort(a)[::-1][:topk].sum())
        except Exception:
            scores.append(0.0)
scores = np.array(scores)
true_bin = np.array([s['is_diseased'] for s in train_s])
h_frac = (true_bin == 0).mean()
pseudo = (scores > np.percentile(scores, h_frac*100)).astype(int)
print(f"Pseudo-label acc: {(pseudo==true_bin).mean():.4f}")

# --- train student ---
loader_t = DataLoader(SimpleDS(train_s, pseudo.tolist(), tf_train),
                      batch_size=64, shuffle=True, num_workers=2)
loader_e = DataLoader(TestDS(test_s, tf_test),
                      batch_size=64, shuffle=False, num_workers=2)

model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
model = model.to(device)
opt   = optim.Adam(model.parameters(), lr=1e-3)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)
crit  = nn.CrossEntropyLoss()

model.train()
for ep in range(30):
    for xb, yb in loader_t:
        xb, yb = xb.to(device), yb.to(device).long()
        loss = crit(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
    sched.step()
    if (ep+1) % 10 == 0: print(f"  Epoch {ep+1}/30")

# --- SAVE per-image probabilities (the new bit) ---
model.eval(); probs = []
with torch.no_grad():
    for xb, _ in loader_e:
        out = model(xb.to(device))
        probs.append(torch.softmax(out, 1)[:, 1].cpu().numpy())
probs  = np.concatenate(probs)
y_true = np.array([s['is_diseased'] for s in test_s])
print(f"Test AUC: {roc_auc_score(y_true, probs):.4f}")

with open(os.path.join(RESULTS_DIR, 'test_probabilities.json'), 'w') as f:
    json.dump({'probs': probs.tolist(), 'y_true': y_true.tolist(),
               'classes': [s['class'] for s in test_s],
               'uahs_heads': sel_heads, 'seed': SEED}, f)

os.makedirs(os.path.join(OUT, 'checkpoints'), exist_ok=True)
torch.save(model.state_dict(), os.path.join(OUT, 'checkpoints', 'student_mobilenetv3.pth'))
print("Saved probabilities + student checkpoint")

In [ ]:
import numpy as np, json, os
import triage_analysis as ta

with open(os.path.join(RESULTS_DIR, 'test_probabilities.json')) as f:
    d = json.load(f)
probs  = np.array(d['probs'])
y_true = np.array(d['y_true'])

print(f"Test set: {len(y_true)} images, {y_true.mean()*100:.1f}% diseased")
rows = ta.triage_table(y_true, probs)
ta.print_triage(rows, label="tea (dataset prevalence)")

# ---- prevalence-adjusted: the REAL deployment condition ----
# Our dataset is disease-enriched because disease datasets are built by
# photographing diseased leaves. A tea garden being scanned is mostly healthy.
def resample_to_prevalence(y, s, target_prev, rng):
    dis = np.where(y == 1)[0]; hea = np.where(y == 0)[0]
    n_dis = int(round(target_prev * len(hea) / (1 - target_prev)))
    n_dis = min(n_dis, len(dis))
    idx = np.r_[hea, rng.choice(dis, n_dis, replace=False)]
    return y[idx], s[idx]

rng = np.random.default_rng(0)
prev_results = {}
for prev in [0.30, 0.20, 0.10]:
    yy, ss = resample_to_prevalence(y_true, probs, prev, rng)
    r = ta.triage_at_sensitivity(yy, ss, 0.95)
    prev_results[prev] = r
    print(f"\n>> Field prevalence {prev*100:.0f}%  (n={r['n_total']}):")
    print(f"   auto-cleared {r['workload_reduction']*100:.1f}% | "
          f"NPV {r['npv_of_cleared']*100:.1f}% | "
          f"missed disease {r['missed_disease_rate']*100:.1f}%")

with open(os.path.join(RESULTS_DIR, 'triage_analysis.json'), 'w') as f:
    json.dump({'dataset_prevalence': rows,
               'field_prevalence': {str(k): v for k, v in prev_results.items()}},
              f, indent=2)
print("\nSaved triage_analysis.json")

In [ ]:
cost_rows = [
    ta.annotation_cost(13084,  5.0, 15.0, "Tea @ 5s/img (fast)"),
    ta.annotation_cost(13084,  8.0, 15.0, "Tea @ 8s/img (typical)"),
    ta.annotation_cost(13084, 15.0, 15.0, "Tea @15s/img (careful)"),
]
ta.print_annotation_cost(cost_rows)

with open(os.path.join(RESULTS_DIR, 'annotation_cost.json'), 'w') as f:
    json.dump(cost_rows, f, indent=2)
print("\nNOTE: replace $15/hr with a CITED Bangladeshi agronomist/extension")
print("officer wage, and report the time as a RANGE, not a point value.")

In [ ]:
!pip install -q onnxscript onnx ai-edge-torch
print("Installed. Now: Runtime -> Restart session, then run Step 2.")

In [ ]:
!pip install -q --force-reinstall torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q "protobuf<6"
print("Restored. Restart the session ONE more time, then verify below.")

In [ ]:
import torch, torchvision
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("cuda:", torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch, torch.nn as nn, os
import torchvision.models as models

OUT  = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
ckpt = os.path.join(OUT, 'checkpoints', 'student_mobilenetv3.pth')
assert os.path.exists(ckpt), "student checkpoint missing — re-run D1"

model_cpu = models.mobilenet_v3_small()
model_cpu.classifier[-1] = nn.Linear(model_cpu.classifier[-1].in_features, 2)
model_cpu.load_state_dict(torch.load(ckpt, map_location='cpu'))
model_cpu.eval()

DEPLOY = os.path.join(OUT, 'deploy'); os.makedirs(DEPLOY, exist_ok=True)
onnx_path = os.path.join(DEPLOY, 'tea_screener.onnx')

torch.onnx.export(model_cpu, torch.randn(1, 3, 224, 224), onnx_path,
                  input_names=['input'], output_names=['logits'],
                  opset_version=13, dynamo=False)   # legacy exporter, no onnxscript

mb = os.path.getsize(onnx_path) / 1e6
print(f"ONNX: {mb:.2f} MB  ->  {onnx_path}")
print(f"vs DINOv1 backbone (86.7 MB): {86.7/mb:.1f}x smaller")

In [ ]:
import numpy as np
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def check(rec):
    try:
        a = np.asarray(Image.open(rec['path']).convert('RGB'))
        return (rec['path'], float(a.std()))
    except Exception:
        return (rec['path'], -1.0)

with ThreadPoolExecutor(16) as ex:
    res = list(tqdm(ex.map(check, all_images), total=len(all_images), desc="verify"))

broken     = [p for p, s in res if s < 0]
suspicious = [p for p, s in res if 0 <= s < 10]   # near-uniform = likely truncated

print(f"\nunreadable            : {len(broken)}")
print(f"suspicious (std < 10) : {len(suspicious)}")
for p in (broken + suspicious)[:10]:
    print("  ", p)

In [ ]:
import numpy as np, os
from PIL import Image
import matplotlib.pyplot as plt

sus = [p for p, s in res if 0 <= s < 10]
print(f"{len(sus)} suspicious files\n")

# stats: is it flat-gray (corrupt) or just dark (a bad photo)?
for p in sus[:12]:
    a = np.asarray(Image.open(p).convert('RGB')).astype(float)
    print(f"mean={a.mean():6.1f}  std={a.std():5.2f}  min={a.min():3.0f}  "
          f"max={a.max():3.0f}  {os.path.basename(p)}")

# LOOK at them — this settles it in one glance
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for ax, p in zip(axes.ravel(), sus[:12]):
    ax.imshow(Image.open(p).convert('RGB'))
    ax.set_title(os.path.basename(p)[:18], fontsize=7)
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch, torch.nn as nn, os
import torchvision.models as models
import onnxscript          # fails loudly here if the install didn't take
print("onnxscript OK")

OUT  = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
ckpt = os.path.join(OUT, 'checkpoints', 'student_mobilenetv3.pth')
assert os.path.exists(ckpt), "student checkpoint missing — re-run D1"

model_cpu = models.mobilenet_v3_small()
model_cpu.classifier[-1] = nn.Linear(model_cpu.classifier[-1].in_features, 2)
model_cpu.load_state_dict(torch.load(ckpt, map_location='cpu'))
model_cpu.eval()

DEPLOY = os.path.join(OUT, 'deploy'); os.makedirs(DEPLOY, exist_ok=True)
dummy = torch.randn(1, 3, 224, 224)

onnx_path = os.path.join(DEPLOY, 'tea_screener.onnx')
torch.onnx.export(model_cpu, dummy, onnx_path,
                  input_names=['input'], output_names=['logits'],
                  opset_version=13, dynamo=False)     # legacy exporter, no onnxscript needed
print(f"ONNX: {os.path.getsize(onnx_path)/1e6:.2f} MB")

try:
    import ai_edge_torch
    edge = ai_edge_torch.convert(model_cpu, (dummy,))
    tfl = os.path.join(DEPLOY, 'tea_screener.tflite')
    edge.export(tfl)
    mb = os.path.getsize(tfl)/1e6
    print(f"TFLite: {mb:.2f} MB  |  {86.7/mb:.1f}x smaller than DINOv1 backbone")
except Exception as e:
    print(f"ai-edge-torch skipped: {e}")
    print("-> ONNX alone is a fine deployment artifact.")

print(f"\nArtifacts -> {DEPLOY}")
!ls -la {DEPLOY}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q onnx onnx-tf tensorflow

import torch, torch.nn as nn, os
import torchvision.models as models

OUT = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"

ckpt = os.path.join(OUT, 'checkpoints', 'student_mobilenetv3.pth')
assert os.path.exists(ckpt), "student checkpoint missing — D1 must be re-run"

model_cpu = models.mobilenet_v3_small()
model_cpu.classifier[-1] = nn.Linear(model_cpu.classifier[-1].in_features, 2)
model_cpu.load_state_dict(torch.load(ckpt, map_location='cpu'))
model_cpu.eval()

DEPLOY = os.path.join(OUT, 'deploy'); os.makedirs(DEPLOY, exist_ok=True)
dummy = torch.randn(1, 3, 224, 224)

onnx_path = os.path.join(DEPLOY, 'tea_screener.onnx')
torch.onnx.export(model_cpu, dummy, onnx_path,
                  input_names=['input'], output_names=['logits'], opset_version=13)
print(f"ONNX: {os.path.getsize(onnx_path)/1e6:.2f} MB")

try:
    import onnx, tensorflow as tf
    from onnx_tf.backend import prepare
    tf_dir = os.path.join(DEPLOY, 'tf_saved')
    prepare(onnx.load(onnx_path)).export_graph(tf_dir)
    conv = tf.lite.TFLiteConverter.from_saved_model(tf_dir)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite = conv.convert()
    tfl = os.path.join(DEPLOY, 'tea_screener_int8.tflite')
    open(tfl, 'wb').write(tflite)
    mb = os.path.getsize(tfl)/1e6
    print(f"TFLite (quantized): {mb:.2f} MB")
    print(f"Compression vs DINOv1 backbone (86.7 MB): {86.7/mb:.1f}x")
except Exception as e:
    print(f"TFLite conversion failed: {e}")
    print("Fallback: ship ONNX and use ONNX Runtime Mobile on Android.")

print(f"\nArtifacts -> {DEPLOY}")

In [ ]:
!pip install -q onnx onnx-tf tensorflow

import torch, torch.nn as nn, os
import torchvision.models as models

if 'OUT' not in globals():
    OUT = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"

model_cpu = models.mobilenet_v3_small()
model_cpu.classifier[-1] = nn.Linear(model_cpu.classifier[-1].in_features, 2)
model_cpu.load_state_dict(torch.load(
    os.path.join(OUT, 'checkpoints', 'student_mobilenetv3.pth'), map_location='cpu'))
model_cpu.eval()

DEPLOY = os.path.join(OUT, 'deploy'); os.makedirs(DEPLOY, exist_ok=True)
dummy = torch.randn(1, 3, 224, 224)

onnx_path = os.path.join(DEPLOY, 'tea_screener.onnx')
torch.onnx.export(model_cpu, dummy, onnx_path,
                  input_names=['input'], output_names=['logits'], opset_version=13)
print(f"ONNX: {os.path.getsize(onnx_path)/1e6:.2f} MB")

try:
    import onnx, tensorflow as tf
    from onnx_tf.backend import prepare
    tf_dir = os.path.join(DEPLOY, 'tf_saved')
    prepare(onnx.load(onnx_path)).export_graph(tf_dir)
    conv = tf.lite.TFLiteConverter.from_saved_model(tf_dir)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]     # dynamic-range int8 quant
    tflite = conv.convert()
    tfl = os.path.join(DEPLOY, 'tea_screener_int8.tflite')
    open(tfl, 'wb').write(tflite)
    mb = os.path.getsize(tfl)/1e6
    print(f"TFLite (quantized): {mb:.2f} MB")
    print(f"Compression vs DINOv1 backbone (86.7 MB): {86.7/mb:.1f}x")
except Exception as e:
    print(f"TFLite conversion failed: {e}")
    print("Fallback: ship ONNX and use ONNX Runtime Mobile on Android.")

print(f"\nArtifacts -> {DEPLOY}")